{
 "cells": [
  {
   "cell_type": "markdown",
   "id": "e3d315d0",
   "metadata": {},
   "source": [
    "# Project 3: RAG-Based AI Teaching Assistant\n",
    "In this project, we will build an AI that can ingest video/audio lectures, understand the content, and answer specific questions based *only* on that lecture. \n",
    "\n",
    "This uses **Retrieval-Augmented Generation (RAG)**.\n",
    "\n",
    "## 1. Setup & Video to Text with Whisper\n",
    "First, we install our dependencies and extract the knowledge from our raw audio/video file. OpenAI's Whisper model converts spoken lectures into highly accurate text."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "5540ec8c",
   "metadata": {},
   "outputs": [],
   "source": [
    "# 1. Install system-level audio processing dependencies\n",
    "!apt-get update && apt-get install -y ffmpeg\n",
    "\n",
    "# 2. Install Python libraries\n",
    "!pip install -q openai-whisper faiss-cpu sentence-transformers langchain openai\n",
    "\n",
    "import os\n",
    "import whisper\n",
    "import numpy as np\n",
    "import faiss\n",
    "from sentence_transformers import SentenceTransformer\n",
    "\n",
    "print(\"✅ All Colab dependencies installed successfully!\\n\")\n",
    "\n",
    "print(\"Loading Whisper Model to GPU...\")\n",
    "# Load the model directly into the GPU (cuda) for massive speedups\n",
    "whisper_model = whisper.load_model(\"base\", device=\"cuda\")\n",
    "\n",
    "# 1. Update the path to point to the uploaded file inside Colab\n",
    "audio_file_path = \"/content/videoplayback.weba\"\n",
    "\n",
    "# 2. Run the actual transcription!\n",
    "print(f\"Transcribing {audio_file_path}... (This might take a moment depending on the file length)\")\n",
    "\n",
    "try:\n",
    "    # Attempt to transcribe the uploaded file\n",
    "    result = whisper_model.transcribe(audio_file_path)\n",
    "    transcript = result[\"text\"]\n",
    "    print(\"\\n✅ Transcription Complete!\")\n",
    "    print(transcript)\n",
    "    \n",
    "except FileNotFoundError:\n",
    "    # If the file hasn't been uploaded yet, use the fallback text\n",
    "    print(f\"\\n❌ Could not find {audio_file_path}. Did you upload it to the Colab files tab?\")\n",
    "    print(\"Using fallback simulated transcript for now...\\n\")\n",
    "    \n",
    "    transcript = \"\"\"\n",
    "    Welcome to Data Science 101. Today we are discussing Machine Learning.\n",
    "    Machine learning is a subset of AI. Supervised learning uses labeled data, like predicting house prices based on square footage. \n",
    "    Unsupervised learning finds hidden patterns, like grouping customers into segments based on purchasing behavior.\n",
    "    We evaluate classification models using accuracy, precision, and recall.\n",
    "    \"\"\"\n",
    "    print(\"✅ Fallback Transcription Complete!\")\n",
    "    print(transcript)"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "3e4e8782",
   "metadata": {},
   "source": [
    "## 2. Chunking & Metadata\n",
    "LLMs have a \"context window\" (a limit to how much text they can read at once). If our lecture is 2 hours long, we can't send the whole transcript to the AI. We must split it into smaller \"chunks\"."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "e8f8d0c0",
   "metadata": {},
   "outputs": [],
   "source": [
    "from langchain.text_splitter import RecursiveCharacterTextSplitter\n",
    "\n",
    "# We split the text into chunks of 100 characters, with 20 characters of overlap to keep context\n",
    "text_splitter = RecursiveCharacterTextSplitter(\n",
    "    chunk_size=100,\n",
    "    chunk_overlap=20,\n",
    "    length_function=len\n",
    ")\n",
    "\n",
    "chunks = text_splitter.split_text(transcript)\n",
    "\n",
    "print(f\"Split the transcript into {len(chunks)} chunks.\\n\")\n",
    "for i, chunk in enumerate(chunks):\n",
    "    print(f\"Chunk {i+1}: {chunk}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "dd839fc4",
   "metadata": {},
   "source": [
    "## 3. Creating Embeddings & Vector Database (FAISS)\n",
    "An **Embedding** translates human text into an array of numbers (vectors). If two sentences have similar meanings, their numbers will be close together in vector space. \n",
    "\n",
    "We will use `SentenceTransformers` to convert our chunks to numbers, and `FAISS` (Facebook AI Similarity Search) to store and search them."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "9a8b9ae6",
   "metadata": {},
   "outputs": [],
   "source": [
    "# Load a free, open-source embedding model\n",
    "print(\"Loading Embedding Model...\")\n",
    "embedding_model = SentenceTransformer('all-MiniLM-L6-v2')\n",
    "\n",
    "# Convert our text chunks into numerical vectors\n",
    "chunk_embeddings = embedding_model.encode(chunks)\n",
    "\n",
    "# Create a FAISS Vector Database\n",
    "dimension = chunk_embeddings.shape[1] # Size of the vectors (384 for this model)\n",
    "vector_db = faiss.IndexFlatL2(dimension)\n",
    "\n",
    "# Add our embeddings to the database\n",
    "vector_db.add(chunk_embeddings)\n",
    "\n",
    "print(f\"✅ Successfully stored {vector_db.ntotal} vectors in the FAISS database.\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "13740d5b",
   "metadata": {},
   "source": [
    "## 4. Matching Chunks (Retrieval)\n",
    "When a student asks a question, we turn their question into a vector, and search the database for the closest matching lecture chunks."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "2075d0cd",
   "metadata": {},
   "outputs": [],
   "source": [
    "# The student's question\n",
    "student_question = \"What is the difference between supervised and unsupervised learning?\"\n",
    "\n",
    "# Convert the question into a vector\n",
    "question_embedding = embedding_model.encode([student_question])\n",
    "\n",
    "# Search the FAISS database for the top 2 most relevant chunks\n",
    "k = 2 \n",
    "distances, indices = vector_db.search(question_embedding, k)\n",
    "\n",
    "print(\"Student Question:\", student_question)\n",
    "print(\"\\n--- Retrieved Context from Lecture ---\")\n",
    "retrieved_context = \"\"\n",
    "for idx in indices[0]:\n",
    "    retrieved_context += chunks[idx] + \" \"\n",
    "    print(f\"- {chunks[idx]}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "1c80bf98",
   "metadata": {},
   "source": [
    "## 5. Prompt Creation & LLM Responses\n",
    "Now we combine the Student's Question and the Retrieved Context into a single Prompt, and send it to a Large Language Model to generate a natural, conversational answer."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "b46f37ba",
   "metadata": {},
   "outputs": [],
   "source": [
    "# Simulated OpenAI API Call \n",
    "# (You would need an actual API key: client = OpenAI(api_key=\"sk-...\"))\n",
    "\n",
    "def ask_teaching_assistant(question, context):\n",
    "    prompt = f\"\"\"\n",
    "    You are a helpful AI Teaching Assistant. Answer the student's question using ONLY the provided lecture context.\n",
    "    If the answer is not in the context, say \"I don't have enough information from the lecture to answer that.\"\n",
    "    \n",
    "    Lecture Context: {context}\n",
    "    \n",
    "    Student Question: {question}\n",
    "    \"\"\"\n",
    "    \n",
    "    # In a real app: \n",
    "    # response = client.chat.completions.create(model=\"gpt-4\", messages=[{\"role\": \"user\", \"content\": prompt}])\n",
    "    # return response.choices[0].message.content\n",
    "    \n",
    "    print(\"\\n[Sending to LLM...]\")\n",
    "    print(prompt)\n",
    "    print(\"\\n--- AI Assistant Response ---\")\n",
    "    print(\"Based on the lecture, supervised learning uses labeled data (like predicting house prices), while unsupervised learning finds hidden patterns (like grouping customers into segments).\")\n",
    "\n",
    "ask_teaching_assistant(student_question, retrieved_context)"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "dcd41a65",
   "metadata": {},
   "source": [
    "## 6. Improving the RAG Pipeline\n",
    "To make this system production-ready, we can upgrade components:\n",
    "1. **Using State-of-the-Art LLMs (GPT-4o / GPT-5):** Newer models have better reasoning capabilities and can handle highly complex student queries without hallucinating.\n",
    "2. **Better Embeddings:** Upgrading from `MiniLM` to `text-embedding-3-large` (OpenAI) captures much deeper semantic meaning.\n",
    "3. **Advanced Retrieval:** Adding a \"Re-ranker\" (like Cohere) to re-score the retrieved chunks ensures the absolute best context is fed to the LLM.\n",
    "4. **Metadata Filtering:** Tagging chunks with timestamps so the AI can say: *\"Supervised learning is explained at 12:04 in the video.\"*"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "datasci",
   "language": "python",
   "name": "datasci"
  },
  "language_info": {
   "name": "python",
   "version": "3.10.20"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}